#Мамоу Асман ИУ5-21М Лаборатораня работа №2

##Блок 1: Импорт библиотек и загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
df = pd.read_csv('healthcare-dataset-stroke-data.csv')
print("Размер датасета:", df.shape)
print(df.info())
print(df.isnull().sum())

Размер датасета: (5110, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB
None
id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0

## Блок 2: Базовая предобработка
Сначала удалим заведомо ненужные столбцы и обработаем пропуски в ключевых признаках (age, embarked, fare).
Целевая переменная: survived.

In [ ]:
# Удаляем столбец 'id' – неинформативный идентификатор
df_clean = df.drop(columns=['id'])

# Заполняем пропуски:
# bmi – медианой (устойчиво к выбросам)
df_clean['bmi'].fillna(df_clean['bmi'].median(), inplace=True)

# Проверим, что пропусков больше нет
print("Пропуски после заполнения:")
print(df_clean.isnull().sum())

# Целевая переменная – stroke
y = df_clean['stroke']

Пропуски после заполнения:
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64


## Блок 3: Масштабирование признаков (три способа)
Сравним StandardScaler, MinMaxScaler, RobustScaler на числовых признаках: age, fare, sibsp, parch.

In [ ]:
# Выбираем числовые признаки (кроме целевой переменной)
num_features = ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease']
X_num = df_clean[num_features].copy()

# Инициализация скейлеров
scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

# Применяем и выводим статистики после масштабирования
for name, scaler in scalers.items():
    X_scaled = scaler.fit_transform(X_num)
    df_scaled = pd.DataFrame(X_scaled, columns=num_features)
    print(f"\n====== {name} ======")
    print("Среднее:\n", df_scaled.mean().round(4))
    print("\nСтандартное отклонение:\n", df_scaled.std().round(4))
    print("\nМин:\n", df_scaled.min().round(4))
    print("\nМакс:\n", df_scaled.max().round(4))


====== StandardScaler ======
Среднее:
 age                  0.0
avg_glucose_level    0.0
bmi                 -0.0
hypertension        -0.0
heart_disease        0.0
dtype: float64

Стандартное отклонение:
 age                  1.0001
avg_glucose_level    1.0001
bmi                  1.0001
hypertension         1.0001
heart_disease        1.0001
dtype: float64

Мин:
 age                 -1.9083
avg_glucose_level   -1.1270
bmi                 -2.4110
hypertension        -0.3286
heart_disease       -0.2389
dtype: float64

Макс:
 age                  1.7148
avg_glucose_level    3.6571
bmi                  8.9284
hypertension         3.0432
heart_disease        4.1850
dtype: float64

====== MinMaxScaler ======
Среднее:
 age                  0.5267
avg_glucose_level    0.2356
bmi                  0.2126
hypertension         0.0975
heart_disease        0.0540
dtype: float64

Стандартное отклонение:
 age                  0.2760
avg_glucose_level    0.2090
bmi                  0.0882
hypertensio

## Блок 4: Обработка выбросов для числовых признаков
4.1 Удаление выбросов – метод межквартильного размаха (IQR) для fare и age.

4.2 Замена выбросов – винсоризация (ограничение значений квантилями 0.01 и 0.99).

In [ ]:
## Блок 4: Обработка выбросов

# 4.1 Удаление выбросов методом IQR (для age, avg_glucose_level, bmi)
def remove_outliers_iqr(df, columns, factor=1.5):
    df_clean = df.copy()
    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - factor * IQR
        upper = Q3 + factor * IQR
        df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]
    return df_clean

df_no_outliers = remove_outliers_iqr(df_clean, columns=['age', 'avg_glucose_level', 'bmi'])
print(f"Исходный размер: {df_clean.shape[0]} строк")
print(f"После удаления выбросов (IQR): {df_no_outliers.shape[0]} строк")

# 4.2 Замена выбросов – винсоризация (ограничение 1% и 99% квантилями)
from scipy.stats import mstats

df_winsor = df_clean.copy()
for col in ['age', 'avg_glucose_level', 'bmi']:
    df_winsor[col] = mstats.winsorize(df_winsor[col], limits=[0.01, 0.01])

print("\n========== Сравнение статистик до и после винсоризации ==========")
print("\nДо винсоризации:")
print(df_clean[['age', 'avg_glucose_level', 'bmi']].describe().round(2))
print("\nПосле винсоризации:")
print(df_winsor[['age', 'avg_glucose_level', 'bmi']].describe().round(2))

Исходный размер: 5110 строк
После удаления выбросов (IQR): 4383 строк

========== Сравнение статистик до и после винсоризации ==========

До винсоризации:
           age  avg_glucose_level      bmi
count  5110.00            5110.00  5110.00
mean     43.23             106.15    28.86
std      22.61              45.28     7.70
min       0.08              55.12    10.30
25%      25.00              77.24    23.80
50%      45.00              91.88    28.10
75%      61.00             114.09    32.80
max      82.00             271.74    97.60

После винсоризации:
           age  avg_glucose_level      bmi
count  5110.00            5110.00  5110.00
mean     43.23             106.05    28.81
std      22.60              44.94     7.41
min       1.08              56.33    15.10
25%      25.00              77.24    23.80
50%      45.00              91.88    28.10
75%      61.00             114.09    32.80
max      82.00             240.71    52.90


## Блок 5: Обработка нестандартного признака (извлечение титула из имени)
Из поля name извлечём титул (Mr, Mrs, Miss, Master и т.д.) и преобразуем в категориальный признак.

In [ ]:
## Блок 5: Обработка нестандартного признака – создание категорий BMI

# Преобразуем непрерывный bmi в категориальный признак 'bmi_category'
def bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif 18.5 <= bmi < 25:
        return 'Normal'
    elif 25 <= bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

df_clean['bmi_category'] = df_clean['bmi'].apply(bmi_category)

print("Распределение нового категориального признака bmi_category:")
print(df_clean['bmi_category'].value_counts())

Распределение нового категориального признака bmi_category:
bmi_category
Obese          1920
Overweight     1610
Normal         1243
Underweight     337
Name: count, dtype: int64


## Блок 6: Подготовка полного набора признаков

In [ ]:
## Блок 6: Подготовка полного набора признаков (пайплайн)

# Определим числовые и категориальные признаки
# Теперь bmi исключим из числовых, т.к. у нас есть его категориальная версия,
num_features = ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease']
cat_features = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status', 'bmi_category']

X = df_clean[num_features + cat_features]
y = df_clean['stroke']

# Числовой пайплайн: медиана + RobustScaler (т.к. есть выбросы)
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# Категориальный пайплайн: мода + OneHotEncoder
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
])

# Применяем
X_prep = preprocessor.fit_transform(X)

# Получаем имена признаков
cat_enc = preprocessor.named_transformers_['cat'].named_steps['encoder']
cat_names = cat_enc.get_feature_names_out(cat_features).tolist()
all_features = num_features + cat_names

X_ready = pd.DataFrame(X_prep, columns=all_features)
print("Размер подготовленных данных:", X_ready.shape)
print("Пример признаков:", X_ready.columns.tolist())

Размер подготовленных данных: (5110, 25)
Пример признаков: ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease', 'gender_Female', 'gender_Male', 'gender_Other', 'ever_married_No', 'ever_married_Yes', 'work_type_Govt_job', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'work_type_children', 'Residence_type_Rural', 'Residence_type_Urban', 'smoking_status_Unknown', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes', 'bmi_category_Normal', 'bmi_category_Obese', 'bmi_category_Overweight', 'bmi_category_Underweight']


## Блок 7: Отбор признаков – Filter method

In [ ]:
## Блок 7: Отбор признаков – Filter method (SelectKBest с mutual_info)

selector_filter = SelectKBest(score_func=mutual_info_classif, k=10)
X_filter = selector_filter.fit_transform(X_ready, y)
selected_mask = selector_filter.get_support()
selected_features_filter = X_ready.columns[selected_mask]

print("Отобранные признаки (Filter, top-10):")
print(selected_features_filter.tolist())

Отобранные признаки (Filter, top-10):
['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease', 'gender_Female', 'ever_married_No', 'ever_married_Yes', 'work_type_children', 'smoking_status_never smoked']


## Блок 8: Отбор признаков – Wrapper method (RFE)

In [ ]:
## Блок 8: Отбор признаков – Wrapper method (RFE с логистической регрессией)

# Используем LogisticRegression с увеличенным max_iter для сходимости
model_lr = LogisticRegression(max_iter=2000, random_state=42)
selector_rfe = RFE(estimator=model_lr, n_features_to_select=10)
selector_rfe.fit(X_ready, y)

selected_features_rfe = X_ready.columns[selector_rfe.support_]
print("Отобранные признаки (RFE, top-10):")
print(selected_features_rfe.tolist())

Отобранные признаки (RFE, top-10):
['age', 'avg_glucose_level', 'hypertension', 'heart_disease', 'ever_married_Yes', 'work_type_Self-employed', 'work_type_children', 'smoking_status_never smoked', 'bmi_category_Overweight', 'bmi_category_Underweight']


## Блок 9: Отбор признаков – Embedded methods (Lasso и RandomForest)

In [ ]:
## Блок 9: Отбор признаков – Embedded methods

# 9.1 Lasso (L1-регуляризация)
model_lasso = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, max_iter=2000, random_state=42)
model_lasso.fit(X_ready, y)

coef = model_lasso.coef_[0]
selected_features_lasso = X_ready.columns[coef != 0]
print("Отобранные признаки (Lasso):")
print(selected_features_lasso.tolist())

# 9.2 RandomForest важность
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_ready, y)

importances = model_rf.feature_importances_
indices = np.argsort(importances)[::-1][:10]
selected_features_rf = X_ready.columns[indices]
print("\nОтобранные признаки (RandomForest importance, top-10):")
print(selected_features_rf.tolist())

Отобранные признаки (Lasso):
['age', 'avg_glucose_level', 'hypertension', 'heart_disease', 'ever_married_Yes', 'work_type_Self-employed', 'Residence_type_Rural', 'smoking_status_never smoked', 'bmi_category_Obese', 'bmi_category_Overweight']

Отобранные признаки (RandomForest importance, top-10):
['avg_glucose_level', 'age', 'bmi', 'hypertension', 'heart_disease', 'Residence_type_Urban', 'work_type_Private', 'Residence_type_Rural', 'gender_Male', 'smoking_status_never smoked']


## Блок 10: Сравнение результатов

In [ ]:
## Блок 10: Сравнение результатов отбора признаков

comparison = pd.DataFrame({
    'Feature': X_ready.columns,
    'Filter': selector_filter.get_support(),
    'RFE': selector_rfe.support_,
    'Lasso': coef != 0,
    'RF_importance': model_rf.feature_importances_
})
comparison['RF_top10'] = comparison['Feature'].isin(selected_features_rf)

# Покажем признаки, которые были выбраны хотя бы одним методом
selected_any = comparison['Filter'] | comparison['RFE'] | comparison['Lasso'] | comparison['RF_top10']
print(comparison[selected_any].sort_values('RF_importance', ascending=False).to_string())

                        Feature  Filter    RFE  Lasso  RF_importance  RF_top10
1             avg_glucose_level    True   True   True       0.251804      True
0                           age    True   True   True       0.227936      True
2                           bmi    True  False  False       0.194828      True
3                  hypertension    True   True   True       0.026208      True
4                 heart_disease    True   True   True       0.025087      True
16         Residence_type_Urban   False  False  False       0.022136      True
12            work_type_Private   False  False  False       0.021763      True
15         Residence_type_Rural   False  False   True       0.021268      True
6                   gender_Male   False  False  False       0.021129      True
19  smoking_status_never smoked    True   True   True       0.020998      True
5                 gender_Female    True  False  False       0.019246     False
13      work_type_Self-employed   False   True   Tru